# CNN Model Training and Evaluation

This notebook trains a baseline CNN using the same generators from `preprocessing.py`.


In [ ]:
import tensorflow as tf
from keras import layers, Model
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import preprocessing

NUM_CLASSES = 4
IMG_SIZE = (224, 224)

def build_cnn_model(num_classes):
    inputs = tf.keras.Input(shape=(224, 224, 3))

    x = layers.Conv2D(32, (3, 3), padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(256, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return Model(inputs, outputs)

cnn_model = build_cnn_model(NUM_CLASSES)
cnn_model.summary()


In [ ]:
cnn_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

cnn_callbacks = [
    EarlyStopping(patience=10, restore_best_weights=True, monitor="val_accuracy"),
    ReduceLROnPlateau(factor=0.5, patience=4, monitor="val_loss", verbose=1),
    ModelCheckpoint("best_cnn_model.keras", save_best_only=True, monitor="val_accuracy")
]

history_cnn = cnn_model.fit(
    preprocessing.train_generator,
    epochs=50,
    validation_data=preprocessing.val_generator,
    callbacks=cnn_callbacks,
)


In [ ]:
from keras.models import load_model

cnn_model = load_model("best_cnn_model.keras")
print(type(cnn_model))
print("CNN model loaded")

preprocessing.val_generator.reset()
loss_cnn, accuracy_cnn = cnn_model.evaluate(preprocessing.val_generator, verbose=1) # type: ignore
print(f"\nCNN Final Val Accuracy : {accuracy_cnn*100:.2f}%")
print(f"CNN Final Val Loss     : {loss_cnn:.4f}")


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

preprocessing.val_generator.reset()
y_pred_probs_cnn = cnn_model.predict(preprocessing.val_generator, verbose=1)
y_pred_cnn = np.argmax(y_pred_probs_cnn, axis=1)
y_true_cnn = preprocessing.val_generator.classes
CLASS_NAMES = list(preprocessing.val_generator.class_indices.keys())

print("\n=== CNN Classification Report ===")
print(classification_report(y_true_cnn, y_pred_cnn, target_names=CLASS_NAMES))

cm_cnn = confusion_matrix(y_true_cnn, y_pred_cnn)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_cnn, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("CNN - Confusion Matrix")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.savefig("cnn_confusion_matrix.png", dpi=150)
plt.show()


# 6- MULTIMODELING (CNN + LSTM)


## 6.1 Bin Routing Text + Tokenizer


In [ ]:
CLASS_TEXT = {
    'cardboard': 'cardboard paper waste goes in the blue recycling bin',
    'glass':     'glass bottle jar waste goes in the green glass bin',
    'metal':     'metal can tin waste goes in the yellow metal bin',
    'plastic':   'plastic bottle container goes in the red plastic bin',
}

CLASS_NAMES = sorted(CLASS_TEXT.keys())
texts = [CLASS_TEXT[c] for c in CLASS_NAMES]

all_words = ' '.join(texts).split()
vocab = ['<PAD>'] + sorted(set(all_words))
word_to_idx = {w: i for i, w in enumerate(vocab)}
VOCAB_SIZE = len(vocab)
MAX_SEQ_LEN = max(len(t.split()) for t in texts)

def encode(text):
    tokens = [word_to_idx[w] for w in text.split()]
    return tokens + [0] * (MAX_SEQ_LEN - len(tokens))

text_padded = np.array([encode(t) for t in texts])
class_to_text = {i: text_padded[i] for i in range(len(CLASS_NAMES))}

print(f"Classes     : {CLASS_NAMES}")
print(f"Vocab size  : {VOCAB_SIZE}")
print(f"Seq length  : {MAX_SEQ_LEN}")
for i, name in enumerate(CLASS_NAMES):
    print(f"  [{i}] {name:<12}  {text_padded[i]}")


## 6.2 Multimodal Generator


In [ ]:
def make_multimodal_generator(image_gen, class_to_text):
    for images, labels in image_gen:
        class_indices = np.argmax(labels, axis=1)
        texts_batch = np.array([class_to_text[idx] for idx in class_indices])
        yield (images, texts_batch), labels


## 6.3 Build Multimodal Model


In [ ]:
from keras.models import load_model

trained_cnn_model = load_model("best_cnn_model.keras")
trained_cnn_model.trainable = False

feature_extractor = Model(
    inputs=trained_cnn_model.input,
    outputs=trained_cnn_model.layers[-2].output,
    name="cnn_features"
)
feature_extractor.trainable = False

image_input = tf.keras.Input(shape=(224, 224, 3), name="image_input")
img_features = feature_extractor(image_input)

text_input = tf.keras.Input(shape=(MAX_SEQ_LEN,), name="text_input")
x = layers.Embedding(VOCAB_SIZE, 64)(text_input)
x = layers.LSTM(128)(x)
x = layers.Dense(128, activation='relu')(x)
text_features = layers.Dropout(0.3)(x)

merged = layers.Concatenate(name="fusion")([img_features, text_features])
merged = layers.Dense(256, activation='relu')(merged)
merged = layers.Dropout(0.4)(merged)
merged = layers.Dense(128, activation='relu')(merged)
merged = layers.Dropout(0.3)(merged)
output = layers.Dense(4, activation='softmax', name="output")(merged)

multimodal_model = Model(
    inputs=[image_input, text_input],
    outputs=output,
    name="multimodal_waste_classifier_cnn"
)

multimodal_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

multimodal_model.summary()


## 6.4 Train


In [ ]:
preprocessing.train_generator.reset()
preprocessing.val_generator.reset()

train_mm = make_multimodal_generator(preprocessing.train_generator, class_to_text)
val_mm = make_multimodal_generator(preprocessing.val_generator, class_to_text)

callbacks = [
    EarlyStopping(patience=10, restore_best_weights=True, monitor="val_accuracy"),
    ModelCheckpoint("best_multimodal_cnn.keras", save_best_only=True, monitor="val_accuracy")
]

history_mm = multimodal_model.fit(
    train_mm,
    steps_per_epoch=len(preprocessing.train_generator),
    epochs=30,
    validation_data=val_mm,
    validation_steps=len(preprocessing.val_generator),
    callbacks=callbacks
)


## 6.5 Confusion Matrix


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

preprocessing.val_generator.reset()
val_mm_pred = make_multimodal_generator(preprocessing.val_generator, class_to_text)

y_pred_probs, y_true = [], []
for (imgs, txts), labels in val_mm_pred:
    preds = multimodal_model.predict([imgs, txts], verbose=0)
    y_pred_probs.extend(preds)
    y_true.extend(np.argmax(labels, axis=1))
    if len(y_true) >= preprocessing.val_generator.samples:
        break

y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.array(y_true)

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Multimodal CNN - Confusion Matrix")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.savefig("confusion_matrix_multimodal_cnn.png", dpi=150)
plt.show()


## 6.6 Inference


In [ ]:
import os

BIN_ROUTING = {
    'cardboard': ('Blue Bin', 'Paper & Cardboard Recycling'),
    'glass':     ('Green Bin', 'Glass Recycling'),
    'metal':     ('Yellow Bin', 'Metal & Cans Recycling'),
    'plastic':   ('Red Bin', 'Plastic Recycling'),
}

def predict_multimodal(img_path):
    img = tf.keras.utils.load_img(img_path, target_size=(224, 224))
    arr = tf.keras.utils.img_to_array(img) / 255.0
    img_batch = np.expand_dims(arr, axis=0)

    all_probs = np.zeros(4)
    for class_idx in range(4):
        text_batch = np.expand_dims(class_to_text[class_idx], axis=0)
        probs = multimodal_model.predict([img_batch, text_batch], verbose=0)[0]
        all_probs += probs
    all_probs /= 4

    predicted_idx = np.argmax(all_probs)
    predicted_class = CLASS_NAMES[predicted_idx]
    confidence = all_probs[predicted_idx]
    bin_name, bin_desc = BIN_ROUTING[predicted_class]

    print(f"\nWaste Type  : {predicted_class.upper()}")
    print(f"Throw in    : {bin_name} - {bin_desc}")
    print(f"Confidence  : {confidence*100:.1f}%")

    plt.figure(figsize=(5, 5))
    plt.imshow(tf.keras.utils.load_img(img_path))
    plt.title(f"{predicted_class.upper()} - {bin_name}\n({confidence*100:.1f}%)")
    plt.axis("off")
    plt.show()

for cls in CLASS_NAMES:
    folder = f"data/raw/{cls}"
    first = os.listdir(folder)[0]
    print(f"{cls:<12}  data/raw/{cls}/{first}")


In [ ]:
predict_multimodal("data/raw/glass/aug_0_1040.jpg")
